In [0]:
%sql
-- Apply PII tags to dim_customers columns
-- First Name
SET TAG ON COLUMN dev.gold.dim_customers.first_name pii = `true`;
SET TAG ON COLUMN dev.gold.dim_customers.first_name pii_type = name;
SET TAG ON COLUMN dev.gold.dim_customers.first_name sensitivity = high;

-- Last Name
SET TAG ON COLUMN dev.gold.dim_customers.last_name pii = `true`;
SET TAG ON COLUMN dev.gold.dim_customers.last_name pii_type = name;
SET TAG ON COLUMN dev.gold.dim_customers.last_name sensitivity = high;

In [0]:
%sql
-- Email
SET TAG ON COLUMN dev.gold.dim_customers.email pii = `true`;
SET TAG ON COLUMN dev.gold.dim_customers.email pii_type = email;
SET TAG ON COLUMN dev.gold.dim_customers.email sensitivity = high;

-- Phone
SET TAG ON COLUMN dev.gold.dim_customers.phone pii = `true`;
SET TAG ON COLUMN dev.gold.dim_customers.phone pii_type = phone;
SET TAG ON COLUMN dev.gold.dim_customers.phone sensitivity = high;

-- Address
SET TAG ON COLUMN dev.gold.dim_customers.address pii = `true`;
SET TAG ON COLUMN dev.gold.dim_customers.address pii_type = address;
SET TAG ON COLUMN dev.gold.dim_customers.address sensitivity = high;

In [0]:
%sql
-- Date of Birth
SET TAG ON COLUMN dev.gold.dim_customers.date_of_birth pii = `true`;
SET TAG ON COLUMN dev.gold.dim_customers.date_of_birth pii_type = dob;
SET TAG ON COLUMN dev.gold.dim_customers.date_of_birth sensitivity = high;

In [0]:
%sql
-- Create a schema for governance functions
CREATE SCHEMA IF NOT EXISTS dev.governance
COMMENT 'Schema for data governance functions and policies';

In [0]:
%sql
-- Masking function for names (shows only first character)
CREATE OR REPLACE FUNCTION dev.governance.mask_name(name STRING)
RETURNS STRING
RETURN CASE 
  WHEN name IS NULL THEN NULL
  WHEN LENGTH(name) = 0 THEN ''
  ELSE CONCAT(SUBSTRING(name, 1, 1), REPEAT('*', GREATEST(LENGTH(name) - 1, 0)))
END;

In [0]:
%sql
-- Masking function for emails (shows domain only)
CREATE OR REPLACE FUNCTION dev.governance.mask_email(email STRING)
RETURNS STRING
RETURN CASE 
  WHEN email IS NULL THEN NULL
  WHEN email LIKE '%@%' THEN CONCAT('***@', SUBSTRING_INDEX(email, '@', -1))
  ELSE '***'
END
;

In [0]:
%sql
-- Masking function for phone numbers (shows last 4 digits)
CREATE OR REPLACE FUNCTION dev.governance.mask_phone(phone BIGINT)
RETURNS BIGINT
RETURN CASE 
  WHEN phone IS NULL THEN NULL
  ELSE CAST(CONCAT('****', RIGHT(CAST(phone AS STRING), 4)) AS BIGINT)
END;

In [0]:
%sql
-- Masking function for addresses (shows city/state only)
CREATE OR REPLACE FUNCTION dev.governance.mask_address(address STRING)
RETURNS STRING
RETURN CASE 
  WHEN address IS NULL THEN NULL
  ELSE '*** [Address Redacted] ***'
END;

In [0]:
%sql
-- Masking function for date of birth (shows only year)
CREATE OR REPLACE FUNCTION dev.governance.mask_dob(dob DATE)
RETURNS STRING
RETURN CASE 
  WHEN dob IS NULL THEN NULL
  ELSE CONCAT(YEAR(dob), '-**-**')
END;

In [0]:
%sql
-- Column mask policy for names
CREATE OR REPLACE POLICY mask_names_policy
ON CATALOG dev
COMMENT 'Mask all name columns across the catalog for non-admin users'
COLUMN MASK dev.governance.mask_name
TO `All Users`
FOR TABLES
MATCH COLUMNS has_tag_value('pii_type', 'name') AS name_col
ON COLUMN name_col;

In [0]:
%sql
-- Column mask policy for emails
CREATE OR REPLACE POLICY mask_emails_policy
ON CATALOG dev
COMMENT 'Mask all email columns across the catalog for non-admin users'
COLUMN MASK dev.governance.mask_email
TO `All Users`
FOR TABLES
MATCH COLUMNS has_tag_value('pii_type', 'email') AS email_col
ON COLUMN email_col;

In [0]:
%sql
-- Column mask policy for phone numbers
CREATE OR REPLACE POLICY mask_phones_policy
ON CATALOG dev
COMMENT 'Mask all phone columns across the catalog for non-admin users'
COLUMN MASK dev.governance.mask_phone
TO `All Users`
FOR TABLES
MATCH COLUMNS has_tag_value('pii_type', 'phone') AS phone_col
ON COLUMN phone_col;

In [0]:
%sql
-- Column mask policy for addresses
CREATE OR REPLACE POLICY mask_addresses_policy
ON CATALOG dev
COMMENT 'Mask all address columns across the catalog for non-admin users'
COLUMN MASK dev.governance.mask_address
TO `All Users`
FOR TABLES
MATCH COLUMNS has_tag_value('pii_type', 'address') AS address_col
ON COLUMN address_col;

In [0]:
%sql
-- Column mask policy for date of birth
CREATE OR REPLACE POLICY mask_dob_policy
ON CATALOG dev
COMMENT 'Mask date of birth columns by showing only year'
COLUMN MASK dev.governance.mask_dob
TO `All Users`
FOR TABLES
MATCH COLUMNS has_tag_value('pii_type', 'dob') AS dob_col
ON COLUMN dob_col;

In [0]:
%sql
-- Verify all policies on the catalog
SHOW EFFECTIVE POLICIES ON CATALOG dev;

In [0]:
%sql
-- Test query to see masked data
SELECT 
  customer_id,
  first_name,
  last_name,
  email,
  phone,
  address,
  date_of_birth
FROM dev.gold.dim_customers
LIMIT 10;

In [0]:
%sql
-- Query to see all PII tagged columns
SELECT
  catalog_name,
  schema_name,
  table_name,
  column_name,
  tag_name,
  tag_value
FROM system.information_schema.column_tags
WHERE catalog_name = 'dev'
  AND schema_name = 'gold'
  AND tag_name IN ('pii', 'pii_type', 'sensitivity')
ORDER BY table_name, column_name, tag_name;

In [0]:
%sql
-- Query to see all PII tagged columns
SELECT
  catalog_name,
  schema_name,
  table_name,
  column_name,
  tag_name,
  tag_value
FROM system.information_schema.column_tags
WHERE catalog_name = 'dev'
  AND schema_name = 'gold'
  AND tag_name IN ('pii', 'pii_type', 'sensitivity')
ORDER BY table_name, column_name, tag_name;